# 第4回 ニューラルネットワーク — 実習ノートブック

**このノートブックは第4回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 直線1本では解けない問題(XOR)に出会う
2. 層を1つ増やすだけで解けるようになるのを見る
3. 中間層の数を変えて、学習の「運」を確かめる
4. 活性化関数を外して、層を重ねる意味が消えるのを見る

所要時間:実習 約20分 / 課題 約40分

## 0. 準備

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import torch, torch.nn as nn, numpy as np
import matplotlib.pyplot as plt, japanize_matplotlib
plt.rcParams["font.size"] = 12
print("準備できました / PyTorch", torch.__version__)

## 1. 4点の問題

2つの入力(0か1)に対して、答えが決まっています。**橙と青を直線1本で分けられますか?**

In [ ]:
# ===== 4点の問題を用意する =====
X = torch.tensor([[0.,0.], [0.,1.], [1.,0.], [1.,1.]])
問題 = {
    "AND": torch.tensor([[0.],[0.],[0.],[1.]]),   # 両方1なら1
    "OR" : torch.tensor([[0.],[1.],[1.],[1.]]),   # どちらか1なら1
    "XOR": torch.tensor([[0.],[1.],[1.],[0.]]),   # 片方だけ1なら1
}

fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
for a, (名, y) in zip(ax, 問題.items()):
    for (x1, x2), t in zip(X.tolist(), y.ravel().tolist()):
        a.scatter(x1, x2, s=260, color="#E08A2B" if t else "#3A46A8", zorder=3)
    a.set_title(名); a.set_xlim(-.3, 1.3); a.set_ylim(-.3, 1.3)
    a.set_xticks([0,1]); a.set_yticks([0,1]); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print("橙と青を、直線1本で分けられますか? XOR だけ引けないはずです。")

## 2. 学習させる準備

In [ ]:
# ===== 学習させる関数(これを使い回します)=====
def 学習(モデル, y, 回数=3000, 学習率=0.1, seed=0):
    torch.manual_seed(seed)
    for m in モデル.modules():                     # 毎回同じ初期値から始める
        if isinstance(m, nn.Linear): m.reset_parameters()
    最適化 = torch.optim.Adam(モデル.parameters(), lr=学習率)
    損失関数 = nn.BCELoss()
    履歴 = []
    for e in range(回数):
        最適化.zero_grad()
        損失 = 損失関数(モデル(X), y)
        損失.backward(); 最適化.step()
        履歴.append(float(損失))
    with torch.no_grad():
        出力 = モデル(X).numpy().ravel()
        正解率 = ((モデル(X) > 0.5).float() == y).float().mean().item()
    return 正解率, 履歴, 出力

def 一層():
    return nn.Sequential(nn.Linear(2,1), nn.Sigmoid())
def 二層(中間=8):
    return nn.Sequential(nn.Linear(2,中間), nn.ReLU(), nn.Linear(中間,1), nn.Sigmoid())

print("準備できました")

## 3. 1層で解いてみる(演習・問1)

ニューロン1個だけのネットワークです。AND・OR・XOR を順に学習させます。

In [ ]:
# ===== 演習・問1:1層で AND / OR / XOR を解く =====
plt.figure(figsize=(8, 4))
for 名, y in 問題.items():
    acc, hist, out = 学習(一層(), y)
    plt.plot(hist, lw=2, label=f"{名}(正解率 {acc:.0%})")
    print(f"{名:>4}: 正解率 {acc:>4.0%}  4点への出力 {np.round(out,3)}")
plt.xlabel("学習回数"); plt.ylabel("損失"); plt.title("1層のネットワークで学習させる")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("\nAND と OR の損失は 0 に向かって下がります。")
print("XOR だけ 0.693 で止まります。4点すべてに 0.5 と答え、「分かりません」と言っている状態です。")

## 4. 層を1つ増やす

入力と出力のあいだに、中間層を1つ挟むだけです。**それ以外は何も変えません。**

In [ ]:
# ===== 1層 vs 2層(XOR)=====
acc1, hist1, out1 = 学習(一層(), 問題["XOR"])
acc2, hist2, out2 = 学習(二層(8), 問題["XOR"])

print(f"1層     : 正解率 {acc1:>4.0%}  出力 {np.round(out1,3)}")
print(f"2層(8個): 正解率 {acc2:>4.0%}  出力 {np.round(out2,3)}")

plt.figure(figsize=(8, 4))
plt.plot(hist1, lw=2.5, color="#C0392B", label=f"1層(正解率 {acc1:.0%})")
plt.plot(hist2, lw=2.5, color="#3A46A8", label=f"2層(正解率 {acc2:.0%})")
plt.xlabel("学習回数"); plt.ylabel("損失"); plt.title("XOR:層を1つ増やすだけで解ける")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("\n違いは中間層があるかないかだけ。データも学習回数も同じです。")

## 5. 中間層は何個必要か(演習・問2)

理論上、XOR を表すのに必要な中間ニューロンは **2個** です。実際はどうでしょうか。

In [ ]:
# ===== 演習・問2:中間層の数を変える =====
print("中間層のニューロン数を変えて、10回ずつ学習させます(初期値の乱数だけ変える)\n")
print(f"{'中間の数':>10}{'正解率100%になった回数':>22}")
結果 = []
for 中間 in [1, 2, 3, 4, 6, 8, 16]:
    成功 = sum(学習(二層(中間), 問題["XOR"], seed=s)[0] == 1.0 for s in range(10))
    結果.append((中間, 成功))
    print(f"{中間:>10}{成功:>18}/10")

plt.figure(figsize=(7.5, 3.8))
plt.bar([str(z[0]) for z in 結果], [z[1] for z in 結果], color="#3A46A8")
plt.xlabel("中間層のニューロン数"); plt.ylabel("10回中の成功回数")
plt.title("理論上は2個で足りるが、実際には8個ほど必要")
plt.grid(alpha=.3, axis="y"); plt.tight_layout(); plt.show()

print("\n理論上、XOR を表せるのに必要な中間ニューロンは2個です。")
print("でも2個では、初期値の運が悪いと途中で行き詰まります(局所解)。")
print("多めに用意しておくと、どれかが当たりを引く確率が上がります。")

## 6. 活性化関数を外すと(演習・問3)

**中間層は8個のまま、ReLU だけを取り除きます。** 予想してから実行してください。

In [ ]:
# ===== 演習・問3:活性化関数を外すとどうなるか =====
def 二層_活性化なし(中間=8):
    return nn.Sequential(nn.Linear(2,中間), nn.Linear(中間,1), nn.Sigmoid())   # ReLU を抜いた

acc_有, _, out_有 = 学習(二層(8), 問題["XOR"])
acc_無, _, out_無 = 学習(二層_活性化なし(8), 問題["XOR"])

print(f"ReLU あり : 正解率 {acc_有:>4.0%}  出力 {np.round(out_有,3)}")
print(f"ReLU なし : 正解率 {acc_無:>4.0%}  出力 {np.round(out_無,3)}")
print()
print("中間層は8個あるのに、ReLU を外すと解けなくなります。")
print("理由:直線を2回通しても、展開すれば1本の直線と同じだからです。")
print("      y = a(bx + c) + d = (ab)x + (ac + d)   ← x の1次式")
print()
print("→ 層を重ねることが効くのは、あいだで「曲げている」からです。")

## 7. 自分で試す — ニューロン1個を設計する

In [ ]:
# ===== 自分で試す(課題1用)=====
# ニューロン1個を手で設計してみる
w1, w2, b = 2.5, -2.0, -0.5      # ← 書きかえてください

def ニューロン(x1, x2):
    合計 = w1*x1 + w2*x2 + b
    return 合計, max(0.0, 合計)   # ReLU

print(f"重み w1={w1}, w2={w2}, バイアス b={b}\n")
print(f"{'x1':>5}{'x2':>5}{'合計':>10}{'ReLU後':>10}   判断")
for x1 in [0, 1]:
    for x2 in [0, 1]:
        合計, y = ニューロン(x1, x2)
        print(f"{x1:>5}{x2:>5}{合計:>10.1f}{y:>10.1f}   {'する' if y > 0 else 'しない'}")
print("\n納得のいく結果になりましたか? ならなければ w1, w2, b を変えてみてください。")
print("それが、まさに「学習」がやっていることです。")

## 8. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:ニューロン1個を設計する
身のまわりの「2つの手がかりから、するかしないかを決める」場面を選んでください。

- (a) 何を決めますか
- (b) x₁ と x₂ は何ですか
- (c) w₁・w₂・b をあなたの感覚で決め、**その理由**を書いてください
- (d) 4通りの入力を計算し、納得のいく結果になるか確かめてください

> 納得がいかなければ重みを調整してください。**それがまさに「学習」がやっていることです。**
> セクション7のセルを書きかえて使えます。

### 課題2:ノートブックを動かして報告する
- (a) 1層と2層で XOR を学習させた損失曲線を重ねた図(セクション4)
- (b) 中間層の数を変えたときの成功回数の表(セクション5)
- (c) 活性化関数を外した実験の結果と、**なぜそうなるかの説明**(3行以上)

### 課題3:次の主張を批判する(300字以上)

> 「ニューラルネットワークは人間の脳を模倣した技術である。脳が複雑な思考をできるのと同じ原理で動いているため、
> 層を十分に深くすれば、人間と同じように考えられるようになるはずだ。」

**全否定しないでください。** 名前の由来は本当に脳から来ていますし、着想を得たのも事実です。
「模倣」という言葉の **どこまでが正しく、どこからが飛躍か** を分けて書いてください。

考える材料:
- 今日見た「中身」は、実際には何をしていましたか
- 飛行機は鳥を参考にしましたが、羽ばたきますか
- 今日の実験で、効いていたのは「深さ」でしたか、それとも別のものでしたか

**加点**:「では、どこまでは言えるのか」まで書けると評価が上がります。